# 步骤五：CodeT5 微调 — Review Comment Generation

In [1]:
import sys
!{sys.executable} -m pip install -r ../requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple --quiet

In [2]:
import sys
import os
import importlib

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

sys.path.append(os.getcwd())
sys.path.insert(0, "..")
import config
importlib.reload(config)

<module 'config' from 'f:\\学习资料\\智能软件工程实践\\lab3\\task3_review_comment\\..\\config.py'>

## 5a. 数据准备 v2：按文件粒度拆分

从 lab1 raw JSON 中提取 PR 信息，按文件粒度拆分：
- 每个样本 = 一个文件的 diff + 对应文件的 review comments
- 解决原方案中整个 PR diff 过长导致截断丢失信息的问题

In [3]:
SKIP_PREPARE = False

from prepare_review_data_v2 import CodeReviewDataPreparerV2

if SKIP_PREPARE and os.path.exists(config.CODEREVIEW_TRAIN_PATH):
    print("SKIP_PREPARE=True，数据已存在，跳过数据准备")
else:
    preparer = CodeReviewDataPreparerV2()
    preparer.run()

代码审查意见生成数据准备 v2：按文件粒度拆分

处理 训练集 数据...
  加载 kubernetes_kubernetes: 300 PRs
  加载 pytorch_pytorch: 300 PRs
  加载 microsoft_vscode: 300 PRs
  加载 langchain-ai_langchain: 300 PRs
  加载 tensorflow_tensorflow: 300 PRs
  训练集: 1253 样本, 跳过无评论: 2581, 跳过无patch: 164, 跳过自动生成: 320, 缺失 PR: 0

处理 验证集 数据...
  验证集: 87 样本, 跳过无评论: 153, 跳过无patch: 0, 跳过自动生成: 1, 缺失 PR: 0

处理 测试集 数据...
  测试集: 104 样本, 跳过无评论: 344, 跳过无patch: 27, 跳过自动生成: 152, 缺失 PR: 0

数据准备完成
  训练集: f:\学习资料\智能软件工程实践\lab3\data\codereview\train.json
  验证集: f:\学习资料\智能软件工程实践\lab3\data\codereview\val.json
  测试集: f:\学习资料\智能软件工程实践\lab3\data\codereview\test.json


## 5b. 序列长度分析（可选）

统计 input 和 target 的 token 长度分布，帮助选择合适的截断长度。

In [4]:
SKIP_ANALYSIS = True

if not SKIP_ANALYSIS:
    from analyze_sequence_length import analyze_length_distribution
    analyze_length_distribution()
else:
    print("SKIP_ANALYSIS=True，跳过序列长度分析")

SKIP_ANALYSIS=True，跳过序列长度分析


## 5c. CodeT5 微调训练

使用 `Salesforce/codet5-base`（Encoder-Decoder 架构），在训练集上微调代码审查意见生成。

**配置：**
- 输入长度: 1024 tokens
- 目标长度: 256 tokens
- Batch size: 1 + 梯度累积 8 步
- 优化器: 8-bit AdamW
- 支持断点续训：自动检测 `checkpoint-*` 并恢复

In [5]:
SKIP_TRAIN = False

if SKIP_TRAIN and os.path.exists(os.path.join(config.CODET5_MODEL_DIR, "model.safetensors")):
    print("SKIP_TRAIN=True，模型已存在，跳过训练")
else:
    from codet5_review import CodeT5ReviewTrainer
    trainer = CodeT5ReviewTrainer()
    trainer.train()

CodeT5 微调：Review Comment Generation
  使用设备: cuda
  GPU: NVIDIA GeForce RTX 4060 Laptop GPU
  显存: 8.6 GB
  输入长度: 1024
  目标长度: 256


e:\Anaconda\envs\ai4se\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


  训练集: 1253 样本
  验证集: 87 样本
  测试集: 104 样本


e:\Anaconda\envs\ai4se\lib\site-packages\accelerate\accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
e:\Anaconda\envs\ai4se\lib\site-packages\torch\nn\modules\module.py:1326: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\cb\pytorch_1000000000000\work\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(



开始训练...

发现有效检查点: f:\学习资料\智能软件工程实践\lab3\data\codebert_models\codet5_review\checkpoint-156，将从此恢复训练...


...

{'eval_loss': 3.020404577255249, 'eval_bleu': 0.0951657022112341, 'eval_rougeL': 0.23388974610457341, 'eval_runtime': 656.9217, 'eval_samples_per_second': 0.132, 'eval_steps_per_second': 0.132, 'epoch': 3.99}
{'loss': 1.1534, 'grad_norm': 2.0331175327301025, 'learning_rate': 1.628048780487805e-05, 'epoch': 4.03}
{'loss': 1.3793, 'grad_norm': 1.851808786392212, 'learning_rate': 1.60191637630662e-05, 'epoch': 4.09}
{'loss': 0.8131, 'grad_norm': 1.803976058959961, 'learning_rate': 1.5757839721254355e-05, 'epoch': 4.15}
{'loss': 1.1652, 'grad_norm': 3.1557199954986572, 'learning_rate': 1.549651567944251e-05, 'epoch': 4.22}
{'loss': 1.0488, 'grad_norm': 2.3697259426116943, 'learning_rate': 1.5235191637630663e-05, 'epoch': 4.28}
{'loss': 1.1362, 'grad_norm': 2.687893867492676, 'learning_rate': 1.4973867595818815e-05, 'epoch': 4.35}
{'loss': 1.0076, 'grad_norm': 1.493849515914917, 'learning_rate': 1.4712543554006969e-05, 'epoch': 4.41}
{'loss': 1.1999, 'grad_norm': 2.0885839462280273, 'learni

  0%|          | 0/87 [00:00<?, ?it/s]

{'eval_loss': 3.018667697906494, 'eval_bleu': 0.07451529189303166, 'eval_rougeL': 0.14429802076864467, 'eval_runtime': 860.6097, 'eval_samples_per_second': 0.101, 'eval_steps_per_second': 0.101, 'epoch': 5.0}


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


{'train_runtime': 4915.2085, 'train_samples_per_second': 2.039, 'train_steps_per_second': 0.254, 'train_loss': 1.1345080542747321, 'epoch': 5.0}

模型已保存到: f:\学习资料\智能软件工程实践\lab3\data\codebert_models\codet5_review


## 5d. 测试集评估

在测试集上计算 BLEU-4 和 ROUGE-L 指标。

In [6]:
if 'trainer' not in dir():
    from codet5_review import CodeT5ReviewTrainer
    trainer = CodeT5ReviewTrainer()
    trainer.load()

results = trainer.evaluate()

print("\n" + "=" * 70)
print("测试集结果汇总")
print("=" * 70)
print(f"  BLEU-4:   {results['bleu']:.4f}")
print(f"  ROUGE-L:  {results['rougeL']:.4f}")


测试集评估 (Review Comment Generation)


  0%|          | 0/104 [00:00<?, ?it/s]

  BLEU-4:   0.0450
  ROUGE-L:  0.0909

测试集结果汇总
  BLEU-4:   0.0450
  ROUGE-L:  0.0909


## 5e. 推理示例

加载模型并对样本生成审查意见。

In [7]:
import json

if 'trainer' not in dir():
    from codet5_review import CodeT5ReviewTrainer
    trainer = CodeT5ReviewTrainer()
    trainer.load()

with open(config.CODEREVIEW_TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

NUM_SAMPLES = 3
print(f"展示 {NUM_SAMPLES} 个测试样本的生成结果:\n")

for i, sample in enumerate(test_data[:NUM_SAMPLES]):
    input_text = sample["input"]
    target_text = sample["target"]
    generated = trainer.generate([input_text])[0]

    print(f"{'='*70}")
    print(f"样本 {i+1}: PR#{sample['pr_id']} ({sample['repo']}) - {sample['filename']}")
    print(f"{'='*70}")
    print(f"[INPUT 前 300 字符]:\n{input_text[:300]}...\n")
    print(f"[参考评论]:\n{target_text[:300]}...\n")
    print(f"[生成评论]:\n{generated[:300]}...\n")

展示 3 个测试样本的生成结果:

样本 1: PR#188211 (pytorch_pytorch) - test/profiler/test_cupti_node_timer.py
[INPUT 前 300 字符]:
title: [profiler][cupti] Add MEMCPY2 (peer-to-peer) record support
body: Stack from [ghstack](https://github.com/ezyang/ghstack/tree/0.15.0) (oldest at bottom):
* __->__ #188211

Subscribe the CUPTI monitor to ActivityKind.MEMCPY2 (CUpti_ActivityMemcpyPtoP4)
so peer-to-peer / cross-device copies (e....

[参考评论]:
Do you need this for NodeTimer observer also?...

[生成评论]:
## ✅ Open SWE Review: No issues found

Open SWE reviewed this PR and found no potential bugs to report.

[Open in Web](https://openswe.vercel.app/agents/reviews/langchain-ai/langchain/3849) • [View Open SWE trace](https://smith.langchain.com/o/ebbaf2eb-769b-4505-aca2-d11de10372a4/projects/p/f2e68aea...

样本 2: PR#188211 (pytorch_pytorch) - torch/profiler/_cupti/cupti_python.py
[INPUT 前 300 字符]:
title: [profiler][cupti] Add MEMCPY2 (peer-to-peer) record support
body: Stack from [ghstack](https://github.com/ezyang/g